In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

data_folder = project_folder / "data"
feature_file = data_folder / "propertylens_featured.csv"

df = pd.read_csv(feature_file)

# Keep variables that are useful for explaining price
analysis_df = df[
    [
        "price_lakh",
        "area_sqft",
        "bhk_group",
        "flat_type",
        "status",
        "rera_approval"
    ]
].copy()

# Remove rows where the values needed for the analysis are invalid
analysis_df = analysis_df[
    (analysis_df["price_lakh"] > 0) &
    (analysis_df["area_sqft"] > 0)
].copy()

# Log transformations help reduce the effect of extreme values
analysis_df["log_price"] = np.log1p(analysis_df["price_lakh"])
analysis_df["log_area"] = np.log1p(analysis_df["area_sqft"])

# Use categorical variables for BHK, property type, status and RERA
model = smf.ols(
    formula="""
    log_price ~ log_area
    + C(bhk_group)
    + C(flat_type)
    + C(status)
    + C(rera_approval)
    """,
    data=analysis_df
).fit(cov_type="HC3")

print("Observations:", len(analysis_df))
print("R-squared:", round(model.rsquared, 3))
print("Adjusted R-squared:", round(model.rsquared_adj, 3))

# Coefficients and significance
results = pd.DataFrame({
    "variable": model.params.index,
    "coefficient": model.params.values,
    "p_value": model.pvalues.values
})

results["coefficient"] = results["coefficient"].round(4)
results["p_value"] = results["p_value"].round(6)

print("\nModel results:")
print(results.to_string(index=False))

Observations: 14225
R-squared: 0.684
Adjusted R-squared: 0.684

Model results:
                                variable  coefficient  p_value
                               Intercept      -3.4764 0.000000
                       C(bhk_group)[T.2]       0.1393 0.002533
                       C(bhk_group)[T.3]       0.3613 0.000000
                       C(bhk_group)[T.4]       0.4454 0.000000
                       C(bhk_group)[T.5]       0.4109 0.000000
                       C(bhk_group)[T.6]       0.4483 0.000003
                       C(bhk_group)[T.7]       0.4367 0.043613
                  C(bhk_group)[T.8+ BHK]      -0.0576 0.611398
           C(bhk_group)[T.Not specified]      -0.0503 0.602723
                   C(flat_type)[T.Floor]      -0.2847 0.000000
                   C(flat_type)[T.House]      -0.1619 0.038692
               C(flat_type)[T.Penthouse]      -0.3617 0.000449
                    C(flat_type)[T.Plot]      -0.3821 0.000000
                   C(flat_type)[T.Villa